## Prerequisites - GPU environment check

**Run the cell below first.** It checks the NVIDIA tools (`nvidia-smi`, `nvcc`, `nsys`, `ncu`) and the Python GPU packages (`numpy`, `numba`, `cupy`) this course needs, and prints how to fix anything missing.

- **OK** - ready to use.
- **MISSING** - a required tool; fix it before running this module.
- **optional** - only affects specific bonus paths; the workbooks skip these gracefully.

In [ ]:
# Prerequisites: verify the GPU toolchain before running this notebook.
# This finds check_gpu.py at the repository root and runs it.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

# GPU-Accelerated Geant4 with TileCal Geometry and Celeritas

> Checked against upstream documentation in May 2026.
>
> Official Celeritas docs: https://celeritas-project.github.io/celeritas/user/index.html
> Celeritas quick start: https://celeritas-project.github.io/celeritas/user/index.html#quick-start-guide
> Geant4 integration examples: https://github.com/celeritas-project/celeritas/tree/main/example/geant4
> DD4hep project page: https://dd4hep.web.cern.ch/dd4hep/
> Geant4 documentation: https://geant4.web.cern.ch/documentation
> TileCal geometry and macro source: https://github.com/celeritas-project/atlas-tilecal-integration
>
> Note: the TileCal repository is still useful for the GDML geometry and macro cards, but its build and integration instructions predate the current Celeritas Geant4 API. In this notebook we use that repository only as a source of geometry and macro files.

This notebook walks through a minimal, current Geant4 plus Celeritas workflow:

1. Install a recent Celeritas toolchain with Spack.
2. Download the TileCal GDML geometry and macro files.
3. Build a small Geant4 application using the current Celeritas tracking-manager integration path.
4. Compare CPU-only and GPU-enabled runs.
5. Inspect the generated Celeritas diagnostics output.

## 0 Environment setup

This notebook targets a **Linux node with an NVIDIA GPU** (the CoDaS-HEP school machines). It cannot run on Windows or on a machine without Geant4/Celeritas: the build and run steps use `%%bash` and a Spack-provided Geant4 + Celeritas stack.

### 0.1 Optional Python environment for Jupyter

If you want an isolated environment for the notebook itself (not required if your kernel already has JupyterLab):

```bash
conda create -n tilegpu -y -c conda-forge python=3.11 jupyterlab
conda activate tilegpu
```

### 0.2 Bootstrap Spack

The **next code cell** clones Spack into `$HOME/spack` (if it is not already there) and initializes it. Spack is the simplest way to install a consistent Geant4 plus Celeritas stack on Linux or WSL2.

Because each `%%bash` cell runs in a fresh shell, every build/run cell below re-sources Spack itself, so you can run this notebook top to bottom.


In [ ]:
%%bash
set -euo pipefail

# 0.2 Bootstrap Spack (one-time clone; safe to re-run).
SPACK_DIR="$HOME/spack"
if [ ! -d "$SPACK_DIR" ]; then
  echo "Cloning Spack into $SPACK_DIR ..."
  git clone --depth=1 https://github.com/spack/spack.git "$SPACK_DIR"
else
  echo "Spack already present at $SPACK_DIR"
fi

# Initialize Spack in this shell and confirm it works.
. "$SPACK_DIR/share/spack/setup-env.sh"
spack --version


## 1 Install Celeritas, Geant4, and DD4hep

The current Celeritas quick-start recommends using Spack for development and integration builds. Geant4 tracking-manager offload requires Geant4 11.0 or newer.

The next cell **auto-detects your NVIDIA GPU** (via `nvidia-smi`) and sets the matching `cuda_arch` automatically. If no GPU is found, it falls back to a CPU-only build. You can override the architecture manually with `spack config add packages:all:variants:"+cuda cuda_arch=<cc>"` if detection is not possible.

Notes:

- `spack install celeritas` brings in Geant4 and the Geant4-facing Celeritas libraries.
- `dd4hep` is included here so the software stack matches the broader detector-description workflow, even though the minimal example below reads GDML directly.
- If you want to force CPU-only runs later, set `CELER_DISABLE_DEVICE=1` in the runtime environment.

### 1.1 Configure an existing installation for this notebook

If Geant4 and Celeritas are already installed, you do not need to reinstall them for every session. The important part is to start Jupyter from a shell where the Celeritas, Geant4, and DD4hep environment is already loaded.

Recommended launch flow:

```bash
conda activate tilegpu
. "$HOME/spack/share/spack/setup-env.sh"
spack load celeritas dd4hep
jupyter lab
```

Before running the build cells in this notebook, verify that the toolchain is visible:

```bash
. "$HOME/spack/share/spack/setup-env.sh"
spack load celeritas dd4hep
which geant4-config
cmake --version
geant4-config --version
spack find --loaded
```

Two practical notes:

- If you open the notebook from VS Code or Jupyter without launching from a preloaded shell, the Python kernel may not see the same Geant4 and Celeritas environment as your terminal.
- The `%%bash` cells in this notebook explicitly source Spack and load the packages again before building so the notebook does not rely on inherited shell state.


In [ ]:
%%bash
set -euo pipefail
. "$HOME/spack/share/spack/setup-env.sh"

# Base compiler settings
spack config add packages:all:variants:"cxxstd=17"

# Detect an NVIDIA GPU and enable a CUDA-enabled build when one is present.
spack external find cuda || true
if command -v nvidia-smi >/dev/null 2>&1; then
  CUDA_CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -n1 | tr -d '. ')
  if [ -n "${CUDA_CC}" ]; then
    echo "Detected NVIDIA GPU (compute capability ${CUDA_CC}); enabling +cuda cuda_arch=${CUDA_CC}"
    spack config add packages:all:variants:"+cuda cuda_arch=${CUDA_CC}"
  else
    echo "nvidia-smi found but could not read compute capability; building CPU-only."
    echo "Set the arch manually if you want GPU support, e.g.:"
    echo '  spack config add packages:all:variants:"+cuda cuda_arch=80"'
  fi
else
  echo "No nvidia-smi found; building a CPU-only Celeritas stack."
fi

# Install the packages used in this lesson (this step can take a while the first time).
spack install celeritas dd4hep
spack load celeritas dd4hep

# Quick sanity checks
geant4-config --version
ddsim --help >/dev/null 2>&1 || true
spack find --loaded


## 2 Download the TileCal geometry and macro files

The upstream TileCal repository still provides useful input files. The macro files we use here are `TBrun.mac` and `TBrun_all.mac`.

## 3 Write a minimal Geant4 plus Celeritas application

The current upstream integration pattern is:

1. Include `CeleritasG4.hh`.
2. Use `TrackingManagerIntegration::Instance()`.
3. Register `TrackingManagerConstructor` on your Geant4 physics list.
4. Call `BeginOfRunAction` and `EndOfRunAction` from a Geant4 run action.
5. Link with `Celeritas::G4` and use `celeritas_target_link_libraries(...)` in CMake.

The code below downloads the TileCal inputs, writes a small Geant4 application, and creates a matching `CMakeLists.txt`. The executable loads the GDML directly through Geant4's GDML parser and executes a Geant4 macro card.

In [ ]:
import pathlib
import urllib.request


tile_repo = "https://raw.githubusercontent.com/celeritas-project/atlas-tilecal-integration/main/"
for filename in (
    "TileTB_2B1EB_nobeamline.gdml",
    "TBrun.mac",
    "TBrun_all.mac",
):
    path = pathlib.Path(filename)
    if not path.exists():
        urllib.request.urlretrieve(tile_repo + filename, path)

cpp_source = r'''#include <memory>
#include <string>

#include <FTFP_BERT.hh>
#include <G4GDMLParser.hh>
#include <G4ParticleGun.hh>
#include <G4ParticleTable.hh>
#include <G4RunManagerFactory.hh>
#include <G4SystemOfUnits.hh>
#include <G4ThreeVector.hh>
#include <G4UImanager.hh>
#include <G4UserRunAction.hh>
#include <G4VUserActionInitialization.hh>
#include <G4VUserDetectorConstruction.hh>
#include <G4VUserPrimaryGeneratorAction.hh>

#include <CeleritasG4.hh>

using TMI = celeritas::TrackingManagerIntegration;

namespace
{
class DetectorConstruction final : public G4VUserDetectorConstruction
{
  public:
    G4VPhysicalVolume* Construct() final
    {
        parser_.Read("TileTB_2B1EB_nobeamline.gdml", false);
        return parser_.GetWorldVolume();
    }

  private:
    G4GDMLParser parser_;
};

class PrimaryGeneratorAction final : public G4VUserPrimaryGeneratorAction
{
  public:
    PrimaryGeneratorAction()
        : gun_(1)
    {
        auto* particle
            = G4ParticleTable::GetParticleTable()->FindParticle("e-");
        gun_.SetParticleDefinition(particle);
        gun_.SetParticleEnergy(18 * GeV);
        gun_.SetParticlePosition(G4ThreeVector{0, 0, 0});
        gun_.SetParticleMomentumDirection(G4ThreeVector{1, 0, 0});
    }

    void GeneratePrimaries(G4Event* event) final
    {
        gun_.GeneratePrimaryVertex(event);
    }

  private:
    G4ParticleGun gun_;
};

class RunAction final : public G4UserRunAction
{
  public:
    void BeginOfRunAction(G4Run const* run) final
    {
        TMI::Instance().BeginOfRunAction(run);
    }

    void EndOfRunAction(G4Run const* run) final
    {
        TMI::Instance().EndOfRunAction(run);
    }
};

class ActionInitialization final : public G4VUserActionInitialization
{
  public:
    void BuildForMaster() const final
    {
        this->SetUserAction(new RunAction{});
    }

    void Build() const final
    {
        this->SetUserAction(new PrimaryGeneratorAction{});
        this->SetUserAction(new RunAction{});
    }
};

celeritas::SetupOptions MakeOptions()
{
    celeritas::SetupOptions options;
    options.max_num_tracks = 4096;
    options.initializer_capacity = 4096 * 128;
    options.ignore_processes = {"CoulombScat"};
    options.output_file = "tile_gpu.out.json";
    return options;
}
}  // namespace

int main(int argc, char* argv[])
{
    std::string macro = argc > 1 ? argv[1] : "TBrun.mac";

    std::unique_ptr<G4RunManager> run_manager{
        G4RunManagerFactory::CreateRunManager(G4RunManagerType::Default)};

    run_manager->SetUserInitialization(new DetectorConstruction{});

    auto& tmi = TMI::Instance();
    auto* physics_list = new FTFP_BERT{/* verbosity = */ 0};
    physics_list->RegisterPhysics(
        new celeritas::TrackingManagerConstructor(&tmi));
    run_manager->SetUserInitialization(physics_list);
    run_manager->SetUserInitialization(new ActionInitialization{});

    tmi.SetOptions(MakeOptions());

    run_manager->Initialize();
    G4UImanager::GetUIpointer()->ApplyCommand("/control/execute " + macro);

    return 0;
}
'''

cmake_source = r'''cmake_minimum_required(VERSION 3.18...4.1)
project(tile_gpu LANGUAGES CXX)

find_package(Celeritas 0.6 REQUIRED)
find_package(Geant4 REQUIRED)

if(NOT CELERITAS_USE_Geant4)
  message(FATAL_ERROR "This Celeritas installation was not built with Geant4 support")
endif()

add_executable(tile_gpu tile_gpu.cc)
target_compile_features(tile_gpu PRIVATE cxx_std_17)

celeritas_target_link_libraries(tile_gpu
  Celeritas::G4
  ${Geant4_LIBRARIES}
)
'''

pathlib.Path("tile_gpu.cc").write_text(cpp_source)
pathlib.Path("CMakeLists.txt").write_text(cmake_source)

print("Prepared input files:")
for path in [
    pathlib.Path("TileTB_2B1EB_nobeamline.gdml"),
    pathlib.Path("TBrun.mac"),
    pathlib.Path("TBrun_all.mac"),
    pathlib.Path("tile_gpu.cc"),
    pathlib.Path("CMakeLists.txt"),
]:
    print(" -", path)
print("CMake is configured for Celeritas 0.6 or newer.")

## 4 Configure and build

This replaces the older hand-written `g++` command that relied on nonstandard environment variables such as `CELERITAS_INCLUDE_DIRS`. The current upstream recommendation is to use CMake plus `find_package(Celeritas ...)`.

Using `0.6` here keeps the example compatible with current stable Celeritas installs while still matching the tracking-manager integration API used in this notebook.

If your Celeritas build includes MPI support and you only want a single-process notebook run, set `CELER_DISABLE_PARALLEL=1` before executing the benchmark cells.

In [ ]:
%%bash
set -euo pipefail
. "$HOME/spack/share/spack/setup-env.sh"
spack load celeritas dd4hep

cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build -j

In [ ]:
import os
import pathlib
import subprocess
import time


def run_tile(macro: str, use_gpu: bool):
    env = os.environ.copy()
    env.setdefault("CELER_DISABLE_PARALLEL", "1")
    if not use_gpu:
        env["CELER_DISABLE_DEVICE"] = "1"
    else:
        env.pop("CELER_DISABLE_DEVICE", None)

    exe = pathlib.Path("build/tile_gpu")
    t0 = time.perf_counter()
    result = subprocess.run(
        [str(exe), macro],
        env=env,
        text=True,
        capture_output=True,
        check=True,
    )
    dt = time.perf_counter() - t0
    return dt, result.stdout, result.stderr


cpu_t, cpu_out, cpu_err = run_tile("TBrun.mac", use_gpu=False)
gpu_t, gpu_out, gpu_err = run_tile("TBrun.mac", use_gpu=True)

print(f"CPU wall time: {cpu_t:.2f} s")
print(f"GPU wall time: {gpu_t:.2f} s")
if gpu_t > 0:
    print(f"Speed-up: {cpu_t / gpu_t:.2f}x")

## 5 Inspect the Celeritas diagnostics output

The example above writes `tile_gpu.out.json`. Its exact contents depend on the Celeritas version and enabled diagnostics, so the safest first step is to inspect the top-level structure rather than hard-code field names.

In [ ]:
import json
import pathlib
from pprint import pprint

path = pathlib.Path("tile_gpu.out.json")
if not path.exists():
    raise FileNotFoundError("Run the executable first so tile_gpu.out.json exists")

with path.open() as handle:
    diagnostics = json.load(handle)

print("Top-level keys:")
for key in sorted(diagnostics):
    print(" -", key)

print("\nSelected preview:")
preview = {key: diagnostics[key] for key in list(diagnostics)[:3]}
pprint(preview)

## 6 Exercises

### Exercise 1: Mixed-particle macro

Run `TBrun_all.mac`, which contains electron, pion, kaon, and proton runs from the upstream TileCal repository.

Questions:

1. Does the overall GPU speed-up decrease compared with `TBrun.mac`?
2. Why is that expected for a workload with a larger hadronic component?

In [ ]:
cpu_t, _, _ = run_tile("TBrun_all.mac", use_gpu=False)
gpu_t, _, _ = run_tile("TBrun_all.mac", use_gpu=True)
print(f"TBrun_all.mac speed-up: {cpu_t / gpu_t:.2f}x")

### Exercise 2: Event-count scaling

Make a copy of `TBrun.mac`, increase the event count, and see whether the GPU run benefits more from the larger workload.

Expected discussion point: GPUs usually amortize setup costs better as the problem size grows, but hadronic-heavy transport can still cap the gain.

Further reading:

- Celeritas user documentation: https://celeritas-project.github.io/celeritas/user/index.html
- Celeritas Geant4 examples: https://github.com/celeritas-project/celeritas/tree/main/example/geant4
- DD4hep project documentation: https://dd4hep.web.cern.ch/dd4hep/
- Geant4 documentation portal: https://geant4.web.cern.ch/documentation
- TileCal geometry repository used for this lesson: https://github.com/celeritas-project/atlas-tilecal-integration

In [ ]:
from pathlib import Path

source = Path("TBrun.mac").read_text()
Path("TBrun_100k.mac").write_text(source.replace("/run/beamOn 10000", "/run/beamOn 100000"))

cpu_t, _, _ = run_tile("TBrun_100k.mac", use_gpu=False)
gpu_t, _, _ = run_tile("TBrun_100k.mac", use_gpu=True)
print(f"TBrun_100k.mac speed-up: {cpu_t / gpu_t:.2f}x")

## 7 Clean-up

In [ ]:
%%bash
rm -rf build
rm -f tile_gpu.cc CMakeLists.txt tile_gpu.out.json TBrun.mac TBrun_all.mac TBrun_100k.mac